# Translate multilingual customer reviews using Snowflake Cortex

This template guides you through translating a sample batch of multilingual customer reviews using Snowflake Cortex in Python and SQL. 

### Context
*Tasty Bytes* is a global food truck network operating in 15 countries with fleet of 450 trucks. They collect customer reviews to get customer feedback on their food-trucks which come in from multiple sources and span multiple languages. 

In this notebook, we will look at how we analyze these collated customer reviews using Snowflake Cortex to understand what our international customers are saying with **Cortex Translate**.

##

### Import sample data to the notebook

In this next SQL query, we will populate sample data that is used in this template.

In [ ]:
%%sql -r IMPORT_SAMPLE_DATA
USE ROLE SNOWFLAKE_LEARNING_ROLE;

-- use the existing database, schema and warehouse
USE DATABASE SNOWFLAKE_LEARNING_DB;

SET schema_name = CONCAT(current_user(), '_TRANSLATE_MULTILINGUAL_CUSTOMER_REVIEWS');
USE SCHEMA IDENTIFIER($schema_name);

  /*--
  • file format and stage creation
  --*/

  CREATE OR REPLACE FILE FORMAT csv_ff 
    TYPE = 'csv';

  CREATE OR REPLACE STAGE s3load
    COMMENT = 'Quickstarts S3 Stage Connection'
    URL = 's3://sfquickstarts/tastybytes-voc/'
    FILE_FORMAT = csv_ff;


  /*--
  • raw zone table build 
  --*/

  -- truck_reviews table
  CREATE OR REPLACE TABLE truck_reviews
  (
      order_id NUMBER(38,0),
      language VARCHAR(16777216),
      source VARCHAR(16777216),
      review VARCHAR(16777216),
      review_id NUMBER(18,0)
  );
  
  /*--
  • raw zone table load 
  --*/
  
  -- truck_reviews table load
  COPY INTO truck_reviews
  FROM @s3load/raw_support/truck_reviews/;

-- setup completion note
SELECT 'Setup is complete' AS note;

**Import python packages**

Snowflake Notebooks include Streamlit and the third-party packages listed in the Snowflake Anaconda channel. 

Now that the necessary packages are installed, we will import the installed packages into the notebook.

In [ ]:
# Import python packages
import pandas as pd

# Snowpark
from snowflake.snowpark.context import get_active_session
import snowflake.snowpark.functions as F
from snowflake.snowpark.functions import when, date_part, col

# Cortex Functions
import snowflake.cortex  as cortex

session = get_active_session()

### Let's preview the reviews
In this next Python cell, we are previewing the data, looking specifically at any non-English reviews.

In [ ]:
reviews_df = (
    session.table('TRUCK_REVIEWS')
    .filter(col('LANGUAGE') != 'en')
)

reviews_df.select("LANGUAGE","REVIEW").show(20, max_width=125)

### Use Cortex Translate on a Python dataframe

In the next cell, we will leverage **Translate** - one of the **Snowflake Cortex specialised LLM functions** available in Snowpark, to translate the multilingual reviews into English to enable easier analysis for anyone who doesn't speak the language of the original review.

In [ ]:
# Conditionally translate reviews that are not english using Cortex Translate
reviews_df = reviews_df.withColumn('TRANSLATED_REVIEW',when(F.col('LANGUAGE') != F.lit("en"), \
                                                            cortex.translate(F.col('REVIEW'), \
                                                                             F.col('LANGUAGE'), \
                                                                             "en")) \
                                   .otherwise(F.col('REVIEW')))

reviews_df.filter(F.col('LANGUAGE') != F.lit("en")) \
.select(["REVIEW","LANGUAGE","TRANSLATED_REVIEW"]).show(20, max_width=75)

## Using Cortex Translate in SQL
Translate can also be completed in SQL by calling `SNOWFLAKE.CORTEX.TRANSLATE`. In this query, we are translating non-English reviews directly into a new `TRANSLATED_REVIEW` column. 

In [ ]:
%%sql -r SQL_TRANSLATE
SELECT 
  REVIEW,
  LANGUAGE,
  CASE 
    WHEN LANGUAGE != 'en' THEN SNOWFLAKE.CORTEX.TRANSLATE(REVIEW, LANGUAGE, 'en')
    ELSE REVIEW
  END AS TRANSLATED_REVIEW
FROM TRUCK_REVIEWS
WHERE LANGUAGE != 'en'
LIMIT 20;

## Conclusion

In this template, we've demonstrated how to translate non-English customer reviews into English using Snowflake's Cortex Translate function. By following these steps, you now have a replicable process for:

- **Identifying non-English reviews:** Filtering and targeting the right data.
- **Applying conditional translation:** Leveraging SQL’s CASE statement to translate text on demand.
- **Generating actionable insights:** Making multilingual data accessible for analysis.

To further enhance your understanding and explore more advanced use cases, continue learning about additional Cortex functions in the [Snowflake Cortex documentation](https://docs.snowflake.com/en/user-guide/snowflake-cortex/llm-functions).


## Visualizations

The following charts explore the structure of the multilingual review dataset — how reviews are distributed across languages and sources, and how review length varies by language.

In [ ]:
%%sql -r lang_counts
SELECT LANGUAGE, COUNT(*) AS REVIEW_COUNT
FROM TRUCK_REVIEWS
WHERE LANGUAGE != 'en'
GROUP BY LANGUAGE
ORDER BY REVIEW_COUNT DESC;

In [ ]:
%%sql -r source_counts
SELECT SOURCE, COUNT(*) AS REVIEW_COUNT
FROM TRUCK_REVIEWS
WHERE LANGUAGE != 'en'
GROUP BY SOURCE
ORDER BY REVIEW_COUNT DESC;

In [ ]:
%%sql -r lang_source_data
SELECT LANGUAGE, SOURCE, COUNT(*) AS COUNT
FROM TRUCK_REVIEWS
WHERE LANGUAGE != 'en'
GROUP BY LANGUAGE, SOURCE
ORDER BY COUNT DESC;

In [ ]:
%%sql -r lengths_data
SELECT LANGUAGE, LENGTH(REVIEW) AS REVIEW_LEN
FROM TRUCK_REVIEWS
WHERE LANGUAGE != 'en';

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Tasty Bytes — Multilingual Review Analytics', fontsize=18, fontweight='bold')

palette = ['#4C72B0','#DD8452','#55A868','#C44E52','#8172B2',
           '#937860','#DA8BC3','#8C8C8C','#CCB974','#64B5CD']

# ── 1. Language Distribution (horizontal bar) ────────────────────────────────
ax1 = axes[0, 0]
ldf = lang_counts.sort_values('REVIEW_COUNT')
bars = ax1.barh(ldf['LANGUAGE'], ldf['REVIEW_COUNT'],
                color=palette[:len(ldf)], edgecolor='white')
ax1.set_xlabel('Number of Reviews', fontsize=11)
ax1.set_title('Reviews by Language (non-English)', fontsize=13, fontweight='bold')
ax1.spines[['top', 'right']].set_visible(False)
for bar in bars:
    w = bar.get_width()
    ax1.text(w + 0.3, bar.get_y() + bar.get_height() / 2,
             str(int(w)), va='center', fontsize=10, fontweight='bold')

# ── 2. Source Donut Chart ─────────────────────────────────────────────────────
ax2 = axes[0, 1]
sdf = source_counts.sort_values('REVIEW_COUNT', ascending=False)
wedges, texts, autotexts = ax2.pie(
    sdf['REVIEW_COUNT'],
    labels=sdf['SOURCE'],
    autopct='%1.1f%%',
    startangle=90,
    colors=palette[:len(sdf)],
    wedgeprops=dict(width=0.5, edgecolor='white'),
    pctdistance=0.75,
    textprops={'fontsize': 10}
)
for at in autotexts:
    at.set_fontweight('bold')
ax2.set_title('Review Sources (non-English)', fontsize=13, fontweight='bold')

# ── 3. Language × Source Heatmap ─────────────────────────────────────────────
ax3 = axes[1, 0]
pivot = lang_source_data.pivot_table(
    index='LANGUAGE', columns='SOURCE', values='COUNT', fill_value=0
)
im = ax3.imshow(pivot.values, cmap='YlOrRd', aspect='auto')
ax3.set_xticks(range(len(pivot.columns)))
ax3.set_xticklabels(pivot.columns, rotation=35, ha='right', fontsize=10)
ax3.set_yticks(range(len(pivot.index)))
ax3.set_yticklabels(pivot.index, fontsize=10)
ax3.set_title('Review Volume: Language × Source', fontsize=13, fontweight='bold')
plt.colorbar(im, ax=ax3, label='Count', shrink=0.8)
max_val = pivot.values.max()
for i in range(len(pivot.index)):
    for j in range(len(pivot.columns)):
        val = int(pivot.values[i, j])
        if val > 0:
            text_color = 'white' if val > max_val * 0.6 else 'black'
            ax3.text(j, i, str(val), ha='center', va='center',
                     fontsize=9, fontweight='bold', color=text_color)

# ── 4. Review Length Box Plot ─────────────────────────────────────────────────
ax4 = axes[1, 1]
languages = sorted(lengths_data['LANGUAGE'].unique())
lang_lengths = [
    lengths_data[lengths_data['LANGUAGE'] == lang]['REVIEW_LEN'].values
    for lang in languages
]
bp = ax4.boxplot(lang_lengths, tick_labels=languages, patch_artist=True,
                 medianprops=dict(color='black', linewidth=2))
for patch, color in zip(bp['boxes'], palette[:len(languages)]):
    patch.set_facecolor(color)
    patch.set_alpha(0.8)
ax4.set_xlabel('Language', fontsize=11)
ax4.set_ylabel('Review Length (characters)', fontsize=11)
ax4.set_title('Review Length Distribution by Language', fontsize=13, fontweight='bold')
ax4.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.show()